Expects the vfi-golden-set + model dataset as input

In [1]:
import os
import cv2
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras

# --- CONFIGURATION ---
MODEL_PATH = "/kaggle/input/2-frame-interpolation-model/checkpoints_backup/vfi_septuplet_epoch_35.keras"
INPUT_DATA_DIR = "/kaggle/input/vfi-golden-set-model/golden_set_septuplets/sequences"
OUTPUT_ROOT = "/kaggle/working/golden_set"

MODEL_INPUT_SIZE = (256, 256)
WEBP_QUALITY = 85 

# --- HELPER FUNCTIONS ---

def get_image_info(path):
    img = cv2.imread(path)
    if img is None: return None
    h, w, _ = img.shape
    return (h, w)

def load_and_preprocess(path):
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_small = cv2.resize(img, MODEL_INPUT_SIZE, interpolation=cv2.INTER_AREA)
    return img_small.astype('float32') / 255.0

def save_optimized_webp(img_array, path, target_dims=None):
    """Saves float32 array as optimized WebP, upscaling if needed."""
    img_uint8 = (img_array * 255.0).clip(0, 255).astype('uint8')
    if target_dims:
        h_orig, w_orig = target_dims
        img_uint8 = cv2.resize(img_uint8, (w_orig, h_orig), interpolation=cv2.INTER_CUBIC)
    
    cv2.imwrite(path, cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_WEBP_QUALITY), WEBP_QUALITY])

def convert_to_webp(src_path, dest_path):
    """Converts a standard image file to WebP."""
    img = cv2.imread(src_path)
    if img is not None:
        cv2.imwrite(dest_path, img, [int(cv2.IMWRITE_WEBP_QUALITY), WEBP_QUALITY])

def process_sequence(model, sequence_folder):
    seq_id = os.path.basename(sequence_folder)
    print(f"📦 Processing Sequence: {seq_id}")

    # Define the new nested paths
    interp_folder = os.path.join(OUTPUT_ROOT, "interpolation", seq_id)
    pred_folder = os.path.join(OUTPUT_ROOT, "prediction", seq_id)
    os.makedirs(interp_folder, exist_ok=True)
    os.makedirs(pred_folder, exist_ok=True)

    # Get original dimensions for the 'Up-Trip'
    first_frame_path = os.path.join(sequence_folder, "im1.jpg")
    orig_dims = get_image_info(first_frame_path)
    if not orig_dims: return

    # Load all 7 frames
    frames_small = []
    for i in range(1, 8):
        f_path = os.path.join(sequence_folder, f"im{i}.jpg")
        img = load_and_preprocess(f_path)
        if img is None: return
        frames_small.append(img)

    # --- 1. INTERPOLATION TASK (Predicting im4) ---
    # Input: 1,2,3,5,6,7
    interp_input = np.concatenate([frames_small[0], frames_small[1], frames_small[2], 
                                   frames_small[4], frames_small[5], frames_small[6]], axis=-1)
    interp_input = np.expand_dims(interp_input, axis=0)
    interp_results = model.predict(interp_input, verbose=0)
    im4_pred = interp_results[1][0] # Head index 1 is interpolation

    # Save all frames to Interpolation folder as WebP
    for i in range(1, 8):
        src = os.path.join(sequence_folder, f"im{i}.jpg")
        dest = os.path.join(interp_folder, f"im{i}.webp")
        convert_to_webp(src, dest)
    save_optimized_webp(im4_pred, os.path.join(interp_folder, "im4_pred.webp"), orig_dims)

    # --- 2. PREDICTION TASK (Predicting im7) ---
    # Input: 1,2,3,4,5,6
    pred_input = np.concatenate(frames_small[0:6], axis=-1)
    pred_input = np.expand_dims(pred_input, axis=0)
    pred_results = model.predict(pred_input, verbose=0)
    im7_pred = pred_results[0][0] # Head index 0 is prediction

    # Save all frames to Prediction folder as WebP
    for i in range(1, 8):
        src = os.path.join(sequence_folder, f"im{i}.jpg")
        dest = os.path.join(pred_folder, f"im{i}.webp")
        convert_to_webp(src, dest)
    save_optimized_webp(im7_pred, os.path.join(pred_folder, "im7_pred.webp"), orig_dims)

def main():
    if os.path.exists(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
    
    print("🧠 Loading Model...")
    model = keras.models.load_model(MODEL_PATH, compile=False)

    sequences = sorted([f.path for f in os.scandir(INPUT_DATA_DIR) if f.is_dir()])
    for seq in sequences:
        process_sequence(model, seq)

    # Zipping utility for easy download
    zip_name = '/kaggle/working/vfi_golden_set_webp'
    print(f"\n🤐 Zipping results into {zip_name}.zip...")
    shutil.make_archive(zip_name, 'zip', OUTPUT_ROOT)
    
    print(f"✨ DONE. Structure optimized for {len(sequences)} sequences.")

if __name__ == "__main__":
    main()

2025-12-19 19:06:19.571684: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766171179.753429      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766171179.805776      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766171180.248523      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766171180.248562      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766171180.248565      55 computation_placer.cc:177] computation placer alr

🧠 Loading Multi-Head Septuplet Model...


I0000 00:00:1766171192.480524      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1766171192.481179      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Processing Sequence: 001...


I0000 00:00:1766171196.099653     130 service.cc:152] XLA service 0x7e7898004540 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766171196.099685     130 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1766171196.099689     130 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1766171196.339114     130 cuda_dnn.cc:529] Loaded cuDNN version 91002
2025-12-19 19:06:38.983467: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-19 19:06:39.177659: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-19 19:06:39.658243: E external/local_xl

Processing Sequence: 002...
Processing Sequence: 003...
Processing Sequence: 004...
Processing Sequence: 005...
Processing Sequence: 006...
Processing Sequence: 007...
Processing Sequence: 008...
Processing Sequence: 009...
Processing Sequence: 010...
Processing Sequence: 011...
Processing Sequence: 012...
Processing Sequence: 013...
Processing Sequence: 014...
Processing Sequence: 015...
Processing Sequence: 016...
Processing Sequence: 017...
Processing Sequence: 018...
Processing Sequence: 019...
Processing Sequence: 020...
Processing Sequence: 021...
Processing Sequence: 022...
Processing Sequence: 023...
Processing Sequence: 024...
Processing Sequence: 025...
Processing Sequence: 026...
Processing Sequence: 027...
Processing Sequence: 028...
Processing Sequence: 029...
Processing Sequence: 030...
Processing Sequence: 031...
Processing Sequence: 032...
Processing Sequence: 033...
Processing Sequence: 034...
Processing Sequence: 035...
Processing Sequence: 036...
Processing Sequence: